# 🎓 SWAL Assistant Fine-Tuning with QLoRA

This notebook fine-tunes a small LLM (Phi-3.5 Mini or Llama 3.2) using SWAL memory data.

**Requirements:**
- Google Colab Pro (for A100 GPU)
- ~10GB storage for model + dataset
- Training time: ~2-4 hours

**Goals:**
- Factuality: 60% → 85%+
- SWAL Context: 40% → 90%+
- Hallucination: 15% → <5%

In [ ]:
# GPU Check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q huggingface_hub

from google.colab import userdata
import os

# HuggingFace token (get from https://huggingface.co/settings/tokens)
# os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
# Upload training data from local or Google Drive
from google.colab import files
import shutil

# Option 1: Upload from local
print("Upload training data (xavier2_training.jsonl):")
uploaded = files.upload()

# Option 2: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/training_data/xavier2_training.jsonl .

In [ ]:
# Load and inspect training data
import json

training_file = "xavier2_training.jsonl"

with open(training_file, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

print(f"Total training examples: {len(data)}")

# Show category distribution
from collections import Counter
categories = Counter([d.get('category', 'unknown') for d in data])
print("\nCategory distribution:")
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

# Show sample
print("\nSample training example:")
sample = data[0]
print(f"  Instruction: {sample['instruction']}")
print(f"  Response: {sample['response'][:100]}...")
print(f"  Category: {sample['category']}")

In [ ]:
# Format data for training
from datasets import Dataset

def format_for_training(example):
    """Format as instruction-response pair."""
    return {
        "text": f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['response']}\n<|end|>"
    }

# Convert to HuggingFace dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(format_for_training, remove_columns=data[0].keys())

print(f"Dataset size: {len(dataset)}")
print(f"Sample formatted text:\n{dataset[0]['text'][:300]}...")

In [ ]:
# Load model with QLoRA config
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "microsoft/Phi-3.5-mini-instruct"  # or "meta-llama/Llama-3.2-3B-Instruct"

# QLoRA config - 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model
print("Loading model with QLoRA...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {model_id}")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training configuration
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./swal-assistant-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optimization="adamw_torch",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    fp16=False,
    bf16=True,
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field="text",
)

In [ ]:
# Start training!
print("Starting training... This will take 2-4 hours.")
print("\nTraining parameters:")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Device: {torch.cuda.get_device_name(0)}")

model.train()
trainer.train()

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./swal-assistant-v1")
tokenizer.save_pretrained("./swal-assistant-v1")

print("Model saved to ./swal-assistant-v1")

In [ ]:
# Test the fine-tuned model
from transformers import pipeline

test_pipe = pipeline(
    "text-generation",
    model="./swal-assistant-v1",
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Test queries
test_queries = [
    "Quién es BELA y qué proyectos tiene SWAL?",
    "Cuál es el precio de ManteniApp?",
    "Quién es Leonardo Duque?",
]

for query in test_queries:
    print(f"\nQ: {query}")
    result = test_pipe(
        f"<|user|>\n{query}\n<|assistant|>\n",
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
    )
    print(f"A: {result[0]['generated_text'].split('<|assistant|>\n')[1]}")

In [ ]:
# Download model to local (optional)
import zipfile
import os

def zip_directory(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.add(file_path, arcname)

print("Zipping model for download...")
zip_directory("./swal-assistant-v1", "./swal-assistant-v1.zip")
print(f"Model size: {os.path.getsize('./swal-assistant-v1.zip') / 1e9:.2f} GB")

# Download
files.download("./swal-assistant-v1.zip")